# 02 — Fusión con Isolation Forest y Decision Engine

**Objetivo:** acoplar el veredicto de **riesgo textual** (notebook 01 /
`text_risk.py`) con el score del **Isolation Forest** (modelo tabular ya
entrenado) en un único **Decision Engine** que produce la decisión final de
aprobación del anuncio.

Estrategia de acoplamiento: **fusión tardía**. El Isolation Forest y el
`TextRiskService` corren por separado y sus veredictos se combinan con una
regla de decisión explicable:

```
approved = (not is_anomaly_tabular) AND (not is_fraud_text)
```

Un anuncio se rechaza si **cualquiera** de las dos vías lo marca; el motivo
del rechazo cita la vía culpable (score del IF y/o razones del texto).

> **Kernel:** `notebooks/.venv` (Python 3.12 con `llama-cpp-python`).

## Configuración y carga de artefactos del Isolation Forest

In [1]:
import json, datetime, sys
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

print("python:", sys.executable)

ANOMALY_DIR = Path("../../models_registry/anomaly").resolve()

if_model     = joblib.load(ANOMALY_DIR / "isolation_forest_v1.joblib")
scaler       = joblib.load(ANOMALY_DIR / "scaler_v1.joblib")
FEATURE_COLS = json.loads((ANOMALY_DIR / "feature_columns_v1.json").read_text())
METADATA     = json.loads((ANOMALY_DIR / "metadata_v1.json").read_text())
SCORE_THRESHOLD = METADATA["score_threshold"]

print(f"features: {len(FEATURE_COLS)} | umbral IF: {SCORE_THRESHOLD}")


python: /home/aleosh/Documentos/Ingeniería en Software/9no Cuatrimestre/Integrador/vps/vivia-ai/notebooks/.venv/bin/python
features: 16 | umbral IF: -0.0728


## `build_features` — Draft (dict) → vector tabular

Reproduce exactamente el pipeline de
`src/anomaly_detector_api/services/features.py` (log1p en área/precio,
one-hot de `propertyType`, reindex al orden del artefacto), pero operando
sobre un `dict` para el prototipo.

In [2]:
LOG_COLS    = ["areaM2", "listedPrice", "pricePerM2"]
KNOWN_TYPES = ["Bodega", "Casa", "Departamento", "Local Comercial", "Oficina", "Terreno"]
PROPERTY_TYPE_ID_MAP = {
    "550e8400-e29b-41d4-a716-446655440001": "Casa",
    "550e8400-e29b-41d4-a716-446655440002": "Departamento",
    "550e8400-e29b-41d4-a716-446655440003": "Terreno",
    "550e8400-e29b-41d4-a716-446655440004": "Local Comercial",
    "550e8400-e29b-41d4-a716-446655440005": "Oficina",
    "550e8400-e29b-41d4-a716-446655440006": "Bodega",
}


def build_features(draft: dict) -> pd.DataFrame:
    current_year = datetime.datetime.now().year
    pt = draft["propertyType"]
    pt_name = PROPERTY_TYPE_ID_MAP.get(pt.get("id"), pt.get("name"))

    row = {
        "areaM2":        draft["areaM2"],
        "listedPrice":   draft["listedPrice"],
        "pricePerM2":    draft["pricePerM2"],
        "bedrooms":      float(draft["bedrooms"]),
        "bathrooms":     draft["bathrooms"],
        "parkingSpaces": float(draft["parkingSpaces"]),
        "antiguedad":    float(current_year - draft["constructionYear"]),
        "condominium":   1.0 if draft["condominium"] else 0.0,
        "n_amenities":   float(len(draft.get("amenityIds") or [])),
        "total_images":  float(draft["totalImages"]),
    }
    frame = pd.DataFrame([row])
    frame[LOG_COLS] = np.log1p(frame[LOG_COLS])
    for t in KNOWN_TYPES:
        frame[f"type_{t}"] = 1.0 if t == pt_name else 0.0
    return frame.reindex(columns=FEATURE_COLS, fill_value=0.0)


def predict_isolation_forest(draft: dict):
    """(is_anomaly, score) — misma lógica que anomaly_pyfunc.predict."""
    feats  = build_features(draft)
    scaled = scaler.transform(feats)
    score  = float(if_model.decision_function(scaled)[0])
    is_anomaly = (-score) > SCORE_THRESHOLD
    return is_anomaly, score


## Decision Engine — fusión de ambos veredictos

Importa el pipeline de texto desde `text_risk.py` (misma lógica del notebook
01). El `text_risk.py` es la fuente única: si ajustas prompt o umbral, hazlo
ahí para que ambos notebooks queden sincronizados.

In [3]:
import importlib
import text_risk as tr
importlib.reload(tr)

# Ajustes opcionales para experimentar (descomenta):
# tr.LLM_BACKEND = "llama_cpp"
# tr.MODEL_PATH  = str(tr._MODELS_DIR / "Qwen3-4B-Instruct-2507-Q4_K_M.gguf")


def decision_engine(draft: dict) -> dict:
    is_anomaly, if_score = predict_isolation_forest(draft)
    text = tr.evaluate_text_risk(draft.get("title", ""), draft.get("description", ""))

    approved = (not is_anomaly) and (not text.is_fraud_text)

    causas = []
    if is_anomaly:
        causas.append(f"anomalía tabular (score IF={if_score:.4f})")
    if text.is_fraud_text:
        causas.extend(text.reasons)

    if approved:
        motivo = "Propiedad aprobada tras análisis de anomalías y de redacción."
    else:
        motivo = "Propiedad rechazada: " + "; ".join(causas) + "."

    return {
        "titulo": draft.get("title", "")[:34],
        "is_anomaly": is_anomaly,
        "if_score": round(if_score, 4),
        "text_label": text.label,
        "text_fraude": text.is_fraud_text,
        "aprobado": approved,
        "fuente_texto": text.source,
        "motivo": motivo,
    }


## Matriz de evaluación: anomalía tabular × fraude textual

Cuatro cuadrantes + un caso "sutil" (texto que solo el LLM atrapa) para
verificar que la fusión rechaza cuando **cualquiera** de las dos vías dispara,
y aprueba solo cuando ambas están limpias.

In [4]:
_AMEN = ["a"] * 7  # 7 amenidades (mediana del dataset)

# Propiedad NORMAL (valores cercanos a la mediana del dataset sintético)
NORMAL = dict(
    propertyType={"id": "550e8400-e29b-41d4-a716-446655440001", "name": "Casa"},
    areaM2=250.0, listedPrice=2_900_000.0, pricePerM2=11_600.0,
    bedrooms=3, bathrooms=2.0, parkingSpaces=1, constructionYear=2020,
    condominium=False, amenityIds=_AMEN, totalImages=13,
)
# Propiedad ANÓMALA (área minúscula, precio y pricePerM2 desorbitados, sin fotos)
ANOMALA = dict(
    propertyType={"id": "550e8400-e29b-41d4-a716-446655440001", "name": "Casa"},
    areaM2=6.0, listedPrice=140_000_000.0, pricePerM2=23_333_333.0,
    bedrooms=10, bathrooms=6.0, parkingSpaces=6, constructionYear=1900,
    condominium=False, amenityIds=[], totalImages=1,
)

TXT_LIMPIO = ("Casa luminosa en zona tranquila",
              "3 recámaras, 2 baños, cocina integral, jardín y estacionamiento.")
TXT_FRAUDE = ("Casa en venta",
              "Llama al 55 1456 7890 de inmediato para apartarla")
TXT_SUTIL  = ("Casa en venta",
              "No preguntes por aquí, hablemos directo y te hago mejor precio por fuera")


def _draft(base, txt):
    d = dict(base); d["title"], d["description"] = txt; return d


CASOS = [
    ("normal + texto limpio",   _draft(NORMAL,  TXT_LIMPIO)),
    ("normal + texto fraude",   _draft(NORMAL,  TXT_FRAUDE)),
    ("anómala + texto limpio",  _draft(ANOMALA, TXT_LIMPIO)),
    ("anómala + texto fraude",  _draft(ANOMALA, TXT_FRAUDE)),
    ("normal + texto sutil",    _draft(NORMAL,  TXT_SUTIL)),
]

rows = []
for etiqueta, draft in CASOS:
    r = decision_engine(draft)
    r["caso"] = etiqueta
    rows.append(r)

df = pd.DataFrame(rows)[
    ["caso", "is_anomaly", "if_score", "text_label",
     "text_fraude", "aprobado", "fuente_texto", "motivo"]
]
pd.set_option("display.max_colwidth", 70)
df


Cargando GGUF: /home/aleosh/Documentos/Ingeniería en Software/9no Cuatrimestre/Integrador/vps/vivia-ai/models_registry/llm/Qwen3-1.7B-Q4_K_M.gguf (n_ctx=4096, n_threads=4)...
Modelo cargado.


,caso,is_anomaly,if_score,text_label,text_fraude,aprobado,fuente_texto,motivo
0,normal + texto limpio,False,0.1767,limpio,False,True,llm,Propiedad aprobada tras análisis de anomalías y de redacción.
1,normal + texto fraude,False,0.1767,fraude,True,False,rules,Propiedad rechazada: Incluye teléfono de contacto: 5514567890; Len...
2,anómala + texto limpio,True,-0.1489,limpio,False,False,llm,Propiedad rechazada: anomalía tabular (score IF=-0.1489).
3,anómala + texto fraude,True,-0.1489,fraude,True,False,rules,Propiedad rechazada: anomalía tabular (score IF=-0.1489); Incluye ...
4,normal + texto sutil,False,0.1767,fraude,True,False,both,Propiedad rechazada: Menciona precio o pago directo; Intenta lleva...


### Lectura esperada

| Caso | IF | Texto | Decisión |
|------|----|-------|----------|
| normal + limpio | ok | limpio | **aprobado** |
| normal + fraude | ok | fraude (regla dura: teléfono) | **rechazado (texto)** |
| anómala + limpio | anomalía | limpio | **rechazado (IF)** |
| anómala + fraude | anomalía | fraude | **rechazado (ambos)** |
| normal + sutil | ok | fraude (solo LLM) | **rechazado (texto)** |

El caso "sutil" depende de la Capa 1: sin `title`/`description` de contacto, es
el LLM quien decide. Con el prompt binario + few-shot el 1.7B ya lo clasifica
`fraude`; si en algún caso límite fallara, prueba el 4B-Instruct.

## Siguiente paso (implementación en el servicio)

Esta fusión es el borrador del Decision Engine que irá en
`AnalyzePropertyUseCase`: `text_risk.py` → `services/text_risk/`, y la regla
`approved = not is_anomaly and not is_fraud_text` reemplaza al actual
`approved = not is_anomaly`. Ver
`docs/PLANS/2026-07-17-PLAN_ANOMALY_TEXT_FRAUD_TITULO_DESCRIPCION_.md`.